# CP_16 — archived capstone notebook

Run from the repository root. Stored outputs are historical evidence, not a fresh execution. CP_20 and CP_21 setup paths were made portable; select your own project kernel. Earlier notebooks retain their original code. See docs/REPRODUCIBILITY.md.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Define the project data locations
initial_data_path = Path("initial_data")
feedback_path = Path("week_4")

# Check the active environment and required files
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

print(
    "Initial data folder found:",
    initial_data_path.is_dir()
)
print(
    "Cumulative inputs found:",
    (feedback_path / "inputs.txt").is_file()
)
print(
    "Cumulative outputs found:",
    (feedback_path / "outputs.txt").is_file()
)

# Stop early if the notebook cannot locate the required data
assert initial_data_path.is_dir(), "Missing initial_data folder."
assert (feedback_path / "inputs.txt").is_file(), "Missing week_4/inputs.txt."
assert (feedback_path / "outputs.txt").is_file(), "Missing week_4/outputs.txt."

Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Cumulative inputs found: True
Cumulative outputs found: True


**Environment and data-path checks**

This cell imports the libraries needed for data handling and displays the
active Python interpreter and working directory. It checks that the initial
data folder and both cumulative Week 4 feedback files are available.

The assertions stop execution if a required path is missing. Successful
checks confirm file availability, but do not yet validate the contents or
update the observations.

The Week 4 files contain four completed rounds. The next cell will rebuild
each function's data set from its initial observations and these four rounds,
avoiding duplicate additions when the notebook is rerun.

In [2]:
import ast
import re

# Parse the cumulative feedback without executing file contents
def read_feedback_rounds(path):
    text = path.read_text(encoding="utf-8-sig").strip()

    # Convert NumPy-style array([...]) representations into plain lists
    text = re.sub(
        r"array\((\[[^\[\]]*\])\)",
        r"\1",
        text
    )

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(f"Unexpected content in {path}")

        rounds.append(ast.literal_eval(node.value))

    return rounds


# Load all four completed rounds
input_rounds = read_feedback_rounds(feedback_path / "inputs.txt")
output_rounds = read_feedback_rounds(feedback_path / "outputs.txt")

assert len(input_rounds) == len(output_rounds) == 4, (
    "Expected four cumulative rounds in each feedback file."
)

for round_inputs, round_outputs in zip(input_rounds, output_rounds):
    assert len(round_inputs) == len(round_outputs) == 8, (
        "Each round must contain one input-output pair per function."
    )

expected_dimensions = {
    1: 2, 2: 2, 3: 3, 4: 4,
    5: 4, 6: 5, 7: 6, 8: 8
}

initial_data = {}
updated_data = {}

for function_number in range(1, 9):
    folder = initial_data_path / f"function_{function_number}"

    # Reload the initial observations on every execution
    X_initial = np.load(
        folder / "initial_inputs.npy",
        allow_pickle=False
    )
    y_initial = np.load(
        folder / "initial_outputs.npy",
        allow_pickle=False
    ).reshape(-1)

    dimensions = expected_dimensions[function_number]

    assert X_initial.ndim == 2
    assert X_initial.shape == (len(y_initial), dimensions)
    assert np.isfinite(X_initial).all()
    assert np.isfinite(y_initial).all()

    # Extract this function's four evaluated query points and outputs
    X_feedback = np.asarray(
        [row[function_number - 1] for row in input_rounds],
        dtype=float
    )
    y_feedback = np.asarray(
        [row[function_number - 1] for row in output_rounds],
        dtype=float
    )

    assert X_feedback.shape == (4, dimensions)
    assert y_feedback.shape == (4,)
    assert np.isfinite(X_feedback).all()
    assert np.isfinite(y_feedback).all()
    assert ((X_feedback >= 0) & (X_feedback < 1)).all()

    # Preserve the initial data and build the cumulative data set
    initial_data[function_number] = {
        "X": X_initial.copy(),
        "y": y_initial.copy()
    }

    updated_data[function_number] = {
        "X": np.vstack([X_initial, X_feedback]),
        "y": np.concatenate([y_initial, y_feedback])
    }

    print(
        f"Function {function_number}: "
        f"{len(y_initial)} → "
        f"{len(updated_data[function_number]['y'])} observations "
        f"| {dimensions} variables"
    )

Function 1: 10 → 14 observations | 2 variables
Function 2: 10 → 14 observations | 2 variables
Function 3: 15 → 19 observations | 3 variables
Function 4: 30 → 34 observations | 4 variables
Function 5: 20 → 24 observations | 4 variables
Function 6: 20 → 24 observations | 5 variables
Function 7: 30 → 34 observations | 6 variables
Function 8: 40 → 44 observations | 8 variables


**Loading and validating four rounds of feedback**

This cell reads the cumulative feedback as literal data, checks that each
of the four rounds contains eight input-output pairs, and validates their
dimensions, finite values and query bounds.

For each function, it rebuilds the data set from the original observations
plus four completed evaluations. Rerunning the cell therefore does not
append the same rounds repeatedly. The initial data are also retained
separately for progress comparisons.

The expected observation counts are 14, 14, 19, 34, 24, 24, 34 and 44 for
Functions 1–8. Only Functions 1 and 2 started with ten observations;
the other functions retain their larger initial data sets.

> *Theory connection: Module 12, Mini-lesson 12.7, PDF pages 35–37,
> describes the iterative query–feedback–update process. These updated
> observations will inform the fifth-round queries.*

In [3]:
from IPython.display import display

# Summarise progress across all four completed rounds
progress_rows = []

for function_number in range(1, 9):
    initial_y = initial_data[function_number]["y"]
    initial_best = float(np.max(initial_y))
    best_so_far = initial_best

    row = {
        "Function": function_number,
        "Initial best": initial_best
    }

    for round_number, round_outputs in enumerate(
        output_rounds, start=1
    ):
        observed_output = float(
            round_outputs[function_number - 1]
        )

        # Compare with the best value available before this round
        improved = observed_output > best_so_far
        improvement = max(0.0, observed_output - best_so_far)

        row[f"Round {round_number} output"] = observed_output
        row[f"Round {round_number} improved"] = improved
        row[f"Round {round_number} improvement"] = improvement

        best_so_far = max(best_so_far, observed_output)

    row["Current best"] = best_so_far
    progress_rows.append(row)

progress_summary = (
    pd.DataFrame(progress_rows)
    .set_index("Function")
)

# Display the latest round against the previous best
round_4_summary = pd.DataFrame(
    {
        "Best before round 4": [
            float(np.max(updated_data[f]["y"][:-1]))
            for f in range(1, 9)
        ],
        "Round 4 output": progress_summary["Round 4 output"],
        "Round 4 improved": progress_summary["Round 4 improved"],
        "Improvement": progress_summary["Round 4 improvement"],
        "Current best": progress_summary["Current best"]
    },
    index=progress_summary.index
)

display(
    round_4_summary.style.format({
        "Best before round 4": "{:.6e}",
        "Round 4 output": "{:.6e}",
        "Improvement": "{:.6e}",
        "Current best": "{:.6e}"
    })
)

# Report which functions improved in each completed round
for round_number in range(1, 5):
    improved_functions = progress_summary.index[
        progress_summary[f"Round {round_number} improved"]
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} "
        f"| Count: {len(improved_functions)}"
    )

,Best before round 4,Round 4 output,Round 4 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,-2.724074e-169,False,0.000000e+00,7.710875e-16
2,6.582395e-01,6.501379e-01,False,0.000000e+00,6.582395e-01
3,-3.483531e-02,-8.912074e-02,False,0.000000e+00,-3.483531e-02
4,1.948721e-01,-5.917795e-01,False,0.000000e+00,1.948721e-01
5,4.215408e+03,1.963044e+03,False,0.000000e+00,4.215408e+03
6,-6.174861e-01,-3.843809e-01,True,2.331052e-01,-3.843809e-01
7,2.097864e+00,6.947609e-01,False,0.000000e+00,2.097864e+00
8,9.850738e+00,9.915965e+00,True,6.522768e-02,9.915965e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2


**Tracking improvements across completed rounds**

This cell compares each observed output with the best value available
before that round. Because every function is a maximisation task, a larger
value represents an improvement, including when the outputs are negative.

The full history is stored in `progress_summary`, while `round_4_summary`
focuses on the latest feedback. The improvement column records the increase
in the best observed value; it is zero when a query does not improve that
best value.

The feedback files show that Functions 6 and 8 improved in round 4.
Function 6 increased from approximately -0.617486 to -0.384381, and
Function 8 increased from approximately 9.850738 to 9.915965.

The other six queries did not improve their previous best values, but their
observations still provide information for refitting the models. The number
of improvements alone is insufficient to judge the overall strategy.

> *Theory connection: Module 12, Mini-lesson 12.7, PDF pages 35–37,
> describes learning from each evaluation and revising the next query.
> Improvement magnitudes should be interpreted within each function,
> because the objective scales differ.*

In [4]:
# Historical predictions recorded in CP_15 before round-4 feedback
# These are the displayed values, retained to their recorded precision.
round_4_predictions = {
    1: {"mean": 1.038813e-05, "std": 8.857243e-04},
    2: {"mean": 6.634182e-01, "std": 5.500270e-02},
    3: {"mean": -6.706418e-02, "std": 7.876711e-02},
    4: {"mean": -3.904580e-01, "std": 4.679397e-01},
    5: {"mean": 4.305302e+03, "std": 2.760754e+02},
    6: {"mean": -6.067783e-01, "std": 1.556102e-01},
    7: {"mean": 1.886739e+00, "std": 2.020933e-01},
    8: {"mean": 9.901145e+00, "std": 7.270987e-02}
}

# Match each historical prediction to its submitted query
round_4_expected_inputs = {
    1: [0.705334, 0.019831],
    2: [0.699810, 0.002104],
    3: [0.933505, 0.993849, 0.852342],
    4: [0.367088, 0.474599, 0.423453, 0.415817],
    5: [0.622329, 0.317899, 0.999131, 0.997718],
    6: [0.551327, 0.429440, 0.664216, 0.837319, 0.028275],
    7: [0.048559, 0.653060, 0.672739, 0.057497, 0.335520, 0.902837],
    8: [
        0.235512, 0.179280, 0.060095, 0.061301,
        0.940810, 0.603515, 0.198305, 0.781321
    ]
}

prediction_rows = []

for function_number in range(1, 9):
    actual_input = np.asarray(
        input_rounds[3][function_number - 1],
        dtype=float
    )

    assert np.allclose(
        actual_input,
        round_4_expected_inputs[function_number],
        rtol=0,
        atol=1e-9
    ), f"Round 4 input mismatch for Function {function_number}"

    prediction = round_4_predictions[function_number]
    predicted_mean = prediction["mean"]
    predicted_std = prediction["std"]

    observed_output = float(
        output_rounds[3][function_number - 1]
    )
    error = observed_output - predicted_mean

    assert predicted_std > 0

    prediction_rows.append({
        "Function": function_number,
        "Predicted mean": predicted_mean,
        "Predicted std": predicted_std,
        "Observed output": observed_output,
        "Observed minus predicted": error,
        "Absolute error / predicted std":
            abs(error) / predicted_std
    })

prediction_review = (
    pd.DataFrame(prediction_rows)
    .set_index("Function")
)

display(
    prediction_review.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "Observed output": "{:.6e}",
        "Observed minus predicted": "{:.6e}",
        "Absolute error / predicted std": "{:.2f}"
    })
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std
Function,,,,,
1,1.038813e-05,8.857243e-04,-2.724074e-169,-1.038813e-05,0.01
2,6.634182e-01,5.500270e-02,6.501379e-01,-1.328027e-02,0.24
3,-6.706418e-02,7.876711e-02,-8.912074e-02,-2.205656e-02,0.28
4,-3.904580e-01,4.679397e-01,-5.917795e-01,-2.013215e-01,0.43
5,4.305302e+03,2.760754e+02,1.963044e+03,-2.342258e+03,8.48
6,-6.067783e-01,1.556102e-01,-3.843809e-01,2.223974e-01,1.43
7,1.886739e+00,2.020933e-01,6.947609e-01,-1.191978e+00,5.90
8,9.901145e+00,7.270987e-02,9.915965e+00,1.482048e-02,0.20


**Comparing round-4 predictions with observed outcomes**

This cell compares the GP predictions recorded before submission with the
actual round-4 feedback. It first verifies that each prediction corresponds
to the input evaluated by the portal.

A positive signed error means that the observed output exceeded the
predicted mean; a negative error means that the model overestimated it.
Dividing the absolute error by the recorded predictive standard deviation
shows how large the discrepancy was relative to the model's uncertainty.

This comparison uses historical predictions rather than predictions from
a model refitted with the new outputs. Refitting first would allow the
model to learn from the observations being assessed.

Large error-to-standard-deviation ratios can flag model mismatch or
underestimated uncertainty. However, one new observation per function
cannot establish whether its uncertainty estimates are well calibrated.

> *Theory connection: Module 12, Mini-lesson 12.1, PDF pages 9–11,
> introduces GP predictions and uncertainty. Module 5, PDF pages 27–31,
> explains why predictive performance must be assessed using observations
> that were not used to fit the evaluated model.*

### Reflection on prediction accuracy — round 4

This cell compares the predictions recorded before the fourth submission
with the outputs subsequently returned by the portal. The comparison
therefore evaluates predictions made before these observations were
available for model fitting.

Functions 1, 2, 3, 4 and 8 had absolute prediction errors below one
predicted standard deviation. However, accurate prediction and
optimisation progress are different: only Functions 6 and 8 improved
their previous best observed outputs in this round.

Function 6 exceeded its predicted mean by approximately 0.2224,
equivalent to 1.43 predicted standard deviations. Function 8 improved
its previous best while remaining close to its predicted mean.

Functions 5 and 7 were substantially overpredicted. Their absolute
errors were 8.48 and 5.90 times their predicted standard deviations,
respectively. Together with the large discrepancies observed in
round 3, these results suggest that predictive uncertainty may be
underestimated in some regions, or that the fitted models do not
adequately represent the response surfaces there.

These observations motivate additional model checks for Functions 5
and 7 before selecting the next queries. Simply widening kernel
bounds or obtaining a higher log-marginal likelihood does not establish
that a model will predict new evaluations more accurately.

For Function 1, the small scaled error should also be interpreted
carefully: predicting another output close to zero does not demonstrate
that the model has located a promising maximum.

> *Methodological note: These ratios are diagnostic indicators, not
> proof that uncertainty estimates are calibrated. Each function
> contributes only one new evaluation in this round, and the queried
> points were deliberately selected rather than sampled randomly.*

In [5]:
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel
)


# Scale inputs and outputs using training data only.
def scaled_regressor(regressor):
    return TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            regressor
        ),
        transformer=StandardScaler()
    )


def make_gp(n_features, upper_bound):
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * n_features,
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )


cv_rows = []
model_templates = {}

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    models = {
        "Training mean": DummyRegressor(strategy="mean"),

        "Linear regression": scaled_regressor(
            LinearRegression()
        ),

        "NN (8,)": scaled_regressor(
            MLPRegressor(
                hidden_layer_sizes=(8,),
                activation="tanh",
                solver="lbfgs",
                alpha=0.1,
                max_iter=2000,
                random_state=42
            )
        ),

        "NN (8, 8)": scaled_regressor(
            MLPRegressor(
                hidden_layer_sizes=(8, 8),
                activation="tanh",
                solver="lbfgs",
                alpha=0.1,
                max_iter=2000,
                random_state=42
            )
        ),

        "GP A_original": make_gp(X.shape[1], 2.0),
        "GP B_wider": make_gp(X.shape[1], 10.0)
    }

    # Store unfitted templates for subsequent model fitting.
    model_templates[function] = models

    # Every model uses exactly the same splits for this function.
    folds = list(
        KFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(X)
    )

    for name, template in models.items():
        training_errors = []
        validation_errors = []

        for train_index, validation_index in folds:
            model = clone(template)

            model.fit(X[train_index], y[train_index])

            training_predictions = model.predict(X[train_index])
            validation_predictions = model.predict(X[validation_index])

            training_errors.append(
                np.sqrt(mean_squared_error(
                    y[train_index], training_predictions
                ))
            )

            validation_errors.append(
                np.sqrt(mean_squared_error(
                    y[validation_index], validation_predictions
                ))
            )

        cv_rows.append({
            "Function": function,
            "Model": name,
            "Mean training RMSE": np.mean(training_errors),
            "Mean CV RMSE": np.mean(validation_errors),
            "CV RMSE standard deviation": np.std(
                validation_errors, ddof=1
            )
        })

    print(f"Function {function}: model comparison completed")


model_comparison = pd.DataFrame(cv_rows)

display(
    model_comparison.style.format({
        "Mean training RMSE": "{:.6e}",
        "Mean CV RMSE": "{:.6e}",
        "CV RMSE standard deviation": "{:.6e}"
    })
)

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pac

Function 1: model comparison completed


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2: model comparison completed


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack

...

Function 7: model comparison completed


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack

Function 8: model comparison completed


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,Function,Model,Mean training RMSE,Mean CV RMSE,CV RMSE standard deviation
0,1,Training mean,7.386166e-04,7.911865e-04,7.115897e-04
1,1,Linear regression,7.155615e-04,8.217648e-04,6.860760e-04
2,1,"NN (8,)",2.263831e-04,1.674256e-03,2.577114e-04
3,1,"NN (8, 8)",1.063397e-04,1.576741e-03,2.879670e-04
4,1,GP A_original,8.388658e-12,1.040310e-03,6.400164e-04
5,1,GP B_wider,8.639462e-11,1.034138e-03,6.389192e-04
6,2,Training mean,2.526229e-01,2.538323e-01,2.594428e-02
7,2,Linear regression,1.933075e-01,2.319255e-01,9.175779e-02
8,2,"NN (8,)",5.672564e-02,2.474243e-01,3.478331e-02
9,2,"NN (8, 8)",5.287138e-02,2.775041e-01,7.246086e-03


### Comparing predictive performance and network complexity

This cell compares six modelling approaches using the observations
available after round 4: a training-mean baseline, linear regression,
two neural networks, and two Gaussian Process configurations.

The neural networks have one hidden layer of eight units and two
hidden layers of eight units each. This comparison tests whether
additional depth is useful for these small tabular data sets.
The two-layer network can represent successive transformations of
the inputs, but its architecture alone does not demonstrate that it
has learned a useful feature hierarchy.

All models use the same three cross-validation splits within each
function. Input and target scaling are fitted separately within each
training fold. Gaussian Process hyperparameters are also fitted using
only the training observations in that fold.

Lower validation RMSE indicates better predictive performance on
the held-out observations. A large difference between training and
validation RMSE can indicate overfitting. The training-mean baseline
helps assess whether a more complex model adds predictive value.

RMSE is expressed in each function's original output units, so its
magnitude should be compared between models for the same function,
rather than across different functions.

> *Interpretation note: These validation scores will inform model
> selection, but they are not an independent final test. The data sets
> are small, the observations include adaptively selected queries,
> and three folds provide limited evidence about performance in
> unexplored regions. A lower average RMSE also does not guarantee
> that a model will identify the best next query.*

> *Connection to Module 16: Comparing network depth with validation
> performance provides a practical way to assess whether additional
> complexity is justified by the available data.*

### Reflection on model comparison

The validation results do not support increasing neural network depth
uniformly across all functions. The two-layer network slightly reduces
mean validation RMSE for Functions 1, 3, 4 and 5, but performs worse
than the one-layer network for Functions 2, 6, 7 and 8. These differences
must be interpreted alongside substantial variation between folds.

Gaussian Processes provide the lowest mean validation errors for
Functions 2, 3, 4 and 8. However, Function 3's GP result is almost
identical to the training-mean baseline, so evidence of useful
predictive structure remains weak. For Function 1, the baseline
outperforms every fitted alternative in this comparison.

Function 5 provides the strongest reason to use a neural network as
an additional surrogate. The one-layer network achieves a mean
validation RMSE of approximately 253, compared with 705 for GP A
and 800 for GP B. The two-layer network reduces this to approximately
248, a small difference relative to the fold-to-fold variation.
I therefore favour the smaller network for this supplementary role.

For Function 6, the one-layer network and GP A perform almost
identically. For Function 7, linear regression has the lowest mean
validation RMSE, while both neural networks fit the training data
closely but perform substantially worse on held-out observations.
This suggests that their flexibility is not justified by the current
evidence for this function.

For Function 8, GP B achieves a mean validation RMSE of approximately
0.200, compared with 0.275 for GP A. This supports retaining the wider
GP configuration as the main surrogate for this function.

These findings motivate a function-specific approach: retain GP-based
uncertainty estimates for query exploration, while using a small
neural network for Function 5 and linear regression for Function 7
as additional checks on predicted performance.

> *Module 16 connection: Extra depth introduces additional
> transformations and parameters, but does not guarantee useful
> hierarchical representations. Model complexity should be justified
> by validation performance and the amount of available data.*

> *Methodological note: These are exploratory model-selection results
> from three folds, not independent test scores or statistical proof
> that one model is superior. The comparison covers the configurations
> listed above, not every kernel setting explored in earlier rounds.*

In [6]:
# Fit reference models using all observations available after round 4.
# These models will be used to assess candidates for round 5.

gp_choices = {
    1: "GP A_original",
    2: "GP A_original",
    3: "GP A_original",
    4: "GP A_original",
    5: "GP A_original",
    6: "GP A_original",
    7: "GP A_original",
    8: "GP B_wider"
}

auxiliary_choices = {
    5: "NN (8,)",
    7: "Linear regression"
}

round_5_gp_models = {}
round_5_auxiliary_models = {}
fitted_model_rows = []

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    gp_name = gp_choices[function]
    gp = clone(model_templates[function][gp_name])
    gp.fit(X, y)

    round_5_gp_models[function] = gp

    auxiliary_name = auxiliary_choices.get(function)

    if auxiliary_name is not None:
        auxiliary = clone(
            model_templates[function][auxiliary_name]
        )
        auxiliary.fit(X, y)
        round_5_auxiliary_models[function] = auxiliary

    fitted_model_rows.append({
        "Function": function,
        "Observations": len(y),
        "GP configuration": gp_name,
        "Fitted kernel": str(gp.kernel_),
        "Auxiliary model": auxiliary_name or "None"
    })

fitted_model_summary = (
    pd.DataFrame(fitted_model_rows)
    .set_index("Function")
)

display(fitted_model_summary)

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pac

,Observations,GP configuration,Fitted kernel,Auxiliary model
Function,,,,
1,14,GP A_original,"1.07**2 * Matern(length_scale=[0.02, 2], nu=2....",None
2,14,GP A_original,"0.93**2 * Matern(length_scale=[0.0649, 2], nu=...",None
3,19,GP A_original,"1.14**2 * Matern(length_scale=[2, 1.65, 0.0587...",None
4,34,GP A_original,"3.05**2 * Matern(length_scale=[1.69, 1.56, 1.6...",None
5,24,GP A_original,"1.42**2 * Matern(length_scale=[0.472, 1.18, 0....","NN (8,)"
6,24,GP A_original,"1.39**2 * Matern(length_scale=[0.525, 0.912, 1...",None
7,34,GP A_original,"0.892**2 * Matern(length_scale=[1.11, 2, 2, 0....",Linear regression
8,44,GP B_wider,"3.09**2 * Matern(length_scale=[3.04, 4.39, 2.3...",None


### Fitting the reference models for round 5

This cell refits the chosen GP configurations on all currently
available observations and stores them for candidate evaluation.
It also fits a supplementary one-layer neural network for Function 5
and a linear regression model for Function 7.

The auxiliary models will help identify agreement or disagreement
with GP predictions. They do not provide the GP predictive standard
deviation, and their predictions will not be treated as interchangeable
uncertainty estimates.

For Functions 1 and 3, retaining a GP supports exploration despite
weak validation evidence. For Functions 5 and 7, GP predictions require
particular scrutiny because of the large errors in previous rounds.

No query has been selected or submitted by this cell. The fitted
models are preparation for comparing new candidate points.

In [7]:
from scipy.spatial import cKDTree

# Compare global exploration with local search around the current best.
beta_round_5 = 2.0

round_5_candidate_proposals = {}
candidate_comparison_rows = []

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    gp = round_5_gp_models[function]

    n_features = X.shape[1]
    best_index = int(np.argmax(y))
    best_input = X[best_index]
    best_output = float(y[best_index])

    rng = np.random.default_rng(42 + function)

    # Global candidates across the permitted domain.
    global_candidates = rng.uniform(
        0.0, 0.999999,
        size=(100000, n_features)
    )

    # Local candidates inside a box extending 0.10 per coordinate
    # around the best observed input, limited by the domain.
    local_lower = np.maximum(0.0, best_input - 0.10)
    local_upper = np.minimum(0.999999, best_input + 0.10)

    local_candidates = rng.uniform(
        local_lower,
        local_upper,
        size=(20000, n_features)
    )

    candidate_pools = {
        "Global": global_candidates,
        "Local": local_candidates
    }

    observation_tree = cKDTree(X)
    round_5_candidate_proposals[function] = {}

    for source, candidates in candidate_pools.items():
        # Evaluate precisely the coordinates that could be submitted.
        candidates = np.unique(
            np.round(candidates, 6),
            axis=0
        )

        distances, _ = observation_tree.query(candidates)

        # Exclude previously evaluated points.
        keep = distances > 1e-6
        candidates = candidates[keep]
        distances = distances[keep]

        if len(candidates) == 0:
            raise ValueError(
                f"No new candidates for Function {function}, {source}."
            )

        # Predict in batches to limit memory use.
        means = []
        standard_deviations = []

        for start in range(0, len(candidates), 5000):
            mean_batch, std_batch = gp.predict(
                candidates[start:start + 5000],
                return_std=True
            )
            means.append(mean_batch)
            standard_deviations.append(std_batch)

        mean = np.concatenate(means)
        std = np.concatenate(standard_deviations)
        ucb = mean + beta_round_5 * std

        selected_index = int(np.argmax(ucb))
        query = candidates[selected_index].copy()

        auxiliary_prediction = np.nan
        if function in round_5_auxiliary_models:
            auxiliary_prediction = float(
                round_5_auxiliary_models[function]
                .predict(query.reshape(1, -1))[0]
            )

        proposal = {
            "query": query,
            "mean": float(mean[selected_index]),
            "std": float(std[selected_index]),
            "ucb": float(ucb[selected_index]),
            "nearest_distance": float(distances[selected_index]),
            "auxiliary_prediction": auxiliary_prediction
        }

        round_5_candidate_proposals[function][source] = proposal

        candidate_comparison_rows.append({
            "Function": function,
            "Search": source,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": best_output,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction
        })

candidate_comparison = (
    pd.DataFrame(candidate_comparison_rows)
    .set_index(["Function", "Search"])
)

display(
    candidate_comparison.style.format({
        "Current best": "{:.6e}",
        "GP mean": "{:.6e}",
        "GP std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

### Comparing global and local candidate searches

This cell generates 100,000 global candidates and 20,000 local
candidates for each function. The local search covers a box extending
up to 0.10 in each coordinate around the best observed input, restricted
to the permitted domain. This neighbourhood size is a heuristic,
not a learned property of the function.

Both candidate pools use the same GP model and acquisition rule:
UCB = predicted mean + 2 × predicted standard deviation.
The highest-scoring candidate from each pool is retained for comparison.

The global search can identify opportunities away from the current
best observation. The local search allocates more candidates to its
neighbourhood. Local search is not pure exploitation: its UCB score
still includes an uncertainty bonus.

Coordinates are rounded to six decimal places before prediction,
and previously evaluated points are excluded. The distance column
reports the Euclidean distance to the nearest existing observation;
it is a geometric diagnostic, not a measure of predictive reliability.

For Functions 5 and 7, the table also reports an auxiliary model's
prediction at each GP-selected candidate. Disagreement can flag
sensitivity to model choice. Agreement does not establish that either
prediction is correct.

> *Interpretation note: The two searches have different candidate
> counts and sampling densities. This is a practical comparison of
> proposals, not a controlled benchmark of search algorithms.
> No final query is selected or submitted by this cell.*

In [8]:
# Select the highest-UCB proposal from the global and local searches.
# Store predictions now, before receiving round 5 feedback.

round_5_selected = {}
round_5_queries = {}
selection_rows = []

for function in range(1, 9):
    proposals = round_5_candidate_proposals[function]

    selected_search = max(
        proposals,
        key=lambda source: proposals[source]["ucb"]
    )

    proposal = proposals[selected_search]
    query = proposal["query"].copy()

    # Validate the exact coordinates that will be submitted.
    X = updated_data[function]["X"]

    assert query.shape == (X.shape[1],)
    assert np.all(np.isfinite(query))
    assert np.all((query >= 0.0) & (query < 1.0))

    nearest_distance = np.min(
        np.linalg.norm(X - query, axis=1)
    )
    assert nearest_distance > 1e-6

    portal_query = "-".join(f"{value:.6f}" for value in query)

    round_5_selected[function] = {
        **proposal,
        "query": query,
        "search": selected_search,
        "model_name": gp_choices[function],
        "model": round_5_gp_models[function],
        "auxiliary_model": round_5_auxiliary_models.get(function),
        "portal_query": portal_query
    }

    round_5_queries[function] = query.copy()

    selection_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "GP configuration": gp_choices[function],
        "Query": portal_query,
        "Predicted mean": proposal["mean"],
        "Predicted std": proposal["std"],
        "UCB": proposal["ucb"],
        "Auxiliary prediction": proposal["auxiliary_prediction"]
    })

round_5_submission = (
    pd.DataFrame(selection_rows)
    .set_index("Function")
)

display(
    round_5_submission.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "UCB": "{:.6e}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

print("\nRound 5 — portal-ready queries:\n")

for function in range(1, 9):
    print(
        f"Function {function}: "
        f"{round_5_selected[function]['portal_query']}"
    )

,Selected search,GP configuration,Query,Predicted mean,Predicted std,UCB,Auxiliary prediction
Function,,,,,,,
1,Global,GP A_original,0.795811-0.503445,-2.299355e-04,9.865848e-04,1.743234e-03,—
2,Global,GP A_original,0.000148-0.977716,2.517883e-01,2.357109e-01,7.232101e-01,—
3,Global,GP A_original,0.999318-0.964676-0.505507,-5.621860e-02,6.718938e-02,7.816017e-02,—
4,Local,GP A_original,0.410712-0.405265-0.384089-0.433507,-8.363377e-02,2.990839e-01,5.145340e-01,—
5,Local,GP A_original,0.596309-0.996343-0.995719-0.998368,4.282913e+03,2.279114e+02,4.738736e+03,4.411068e+03
6,Global,GP A_original,0.548648-0.528095-0.973039-0.990992-0.036642,-5.450000e-01,1.717249e-01,-2.015501e-01,—
7,Local,GP A_original,0.006389-0.227533-0.638856-0.045721-0.344255-0.779122,2.030465e+00,1.377804e-01,2.306026e+00,7.397378e-01
8,Local,GP B_wider,0.167821-0.083323-0.143724-0.015688-0.998566-0.507939-0.294815-0.865301,1.000922e+01,6.393926e-02,1.013710e+01,—



Round 5 — portal-ready queries:

Function 1: 0.795811-0.503445
Function 2: 0.000148-0.977716
Function 3: 0.999318-0.964676-0.505507
Function 4: 0.410712-0.405265-0.384089-0.433507
Function 5: 0.596309-0.996343-0.995719-0.998368
Function 6: 0.548648-0.528095-0.973039-0.990992-0.036642
Function 7: 0.006389-0.227533-0.638856-0.045721-0.344255-0.779122
Function 8: 0.167821-0.083323-0.143724-0.015688-0.998566-0.507939-0.294815-0.865301


### Final query selection — round 5

For each function, I selected the proposal with the higher UCB score
between the global and local candidate pools. The exploration weight
remained fixed at 2.0.

The selected proposals come from the global search for Functions
1, 2, 3 and 6, and from the local search for Functions 4, 5, 7 and 8.
These labels describe where candidates were sampled, rather than
whether a query is purely exploratory or exploitative.

For Function 2, the global proposal has a substantially lower predicted
mean than the local proposal, but a slightly higher UCB because of its
larger uncertainty. Function 6 also favours the global proposal through
its uncertainty bonus. These selections deliberately accept possible
short-term losses to investigate less familiar regions.

For Function 5, both the GP and the supplementary neural network
predict an output above the current best at the selected local point.
Their agreement supports investigating this neighbourhood, but does
not establish that an improvement will occur.

For Function 7, substantial disagreement remains: the GP predicts
approximately 2.03, whereas linear regression predicts approximately
0.74. I retain the local UCB proposal as an experiment near the best
observed input, while recording the disagreement for later evaluation.
Proximity to an existing observation does not guarantee reliability.

For Function 8, the selected local proposal has both a higher predicted
mean and a higher UCB than the global alternative. It therefore offers
a way to investigate further improvement near the current best input.

The auxiliary models were used as diagnostics rather than as filters
or components of the acquisition score. Final selection still follows
the GP-UCB rule.

This cell stores the selected coordinates, model references and
pre-submission predictions in notebook memory. It does not submit
anything to the portal. These predictions will be compared with the
actual outputs when round 5 feedback becomes available.

> *Interpretation note: UCB is a model-based acquisition score, not
> a guaranteed upper bound on the unknown output. The large errors
> observed previously for Functions 5 and 7 remain relevant when
> interpreting these proposals.*

In [9]:
# Display the final round 5 queries in portal-ready format.
final_queries_table = pd.DataFrame([
    {
        "Function": function,
        "Query for submission": round_5_selected[function]["portal_query"]
    }
    for function in range(1, 9)
]).set_index("Function")

# Display full query strings without truncation.
with pd.option_context("display.max_colwidth", None):
    display(final_queries_table)

,Query for submission
Function,
1,0.795811-0.503445
2,0.000148-0.977716
3,0.999318-0.964676-0.505507
4,0.410712-0.405265-0.384089-0.433507
5,0.596309-0.996343-0.995719-0.998368
6,0.548648-0.528095-0.973039-0.990992-0.036642
7,0.006389-0.227533-0.638856-0.045721-0.344255-0.779122
8,0.167821-0.083323-0.143724-0.015688-0.998566-0.507939-0.294815-0.865301


This table presents the eight selected queries for round 5 in the
required portal format. Each coordinate has six decimal places,
with coordinates separated by hyphens. The values are taken directly
from the stored selections to avoid transcription errors.
